# 03: LLM Function Calling, Tool Orchestration & JSON Schemas

**Track 13: Generative AI, LLMs, RAG & Multi-Agent Swarms** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Structured tool calling for LLMs: Pydantic schemas, OpenAI-compatible function descriptors, execution dispatch loops, and validation error handling.


## 1. Tool Declaration & Pydantic Schema Specification
Define structured tools for an autonomous agent.

In [ ]:
import json

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculate_mortgage",
            "description": "Calculates monthly mortgage payment for a property",
            "parameters": {
                "type": "object",
                "properties": {
                    "loan_amount": {"type": "number", "description": "Principal loan in USD"},
                    "annual_rate": {"type": "number", "description": "Interest rate e.g. 0.065 for 6.5%"},
                    "term_years": {"type": "integer", "description": "Loan duration in years"}
                },
                "required": ["loan_amount", "annual_rate", "term_years"]
            }
        }
    }
]

def calculate_mortgage(loan_amount: float, annual_rate: float, term_years: int) -> float:
    monthly_rate = annual_rate / 12.0
    n_payments = term_years * 12
    payment = loan_amount * (monthly_rate * (1 + monthly_rate)**n_payments) / ((1 + monthly_rate)**n_payments - 1)
    return round(payment, 2)

tool_registry = {"calculate_mortgage": calculate_mortgage}

def dispatch_tool_call(tool_name: str, arguments_json: str):
    args = json.loads(arguments_json)
    if tool_name in tool_registry:
        return tool_registry[tool_name](**args)
    raise ValueError(f"Tool {tool_name} not found")

sample_call = {"name": "calculate_mortgage", "arguments": json.dumps({"loan_amount": 400000, "annual_rate": 0.065, "term_years": 30})}
res = dispatch_tool_call(sample_call["name"], sample_call["arguments"])
print(f"Tool Result: Monthly Payment = ${res}")